# IndoNLI — Human Text Extraction
Notebook ini mengekstrak kolom `premise` dari dataset IndoNLI sebagai **human-written samples**.

**Output:**
- `data/raw/indonli_human.jsonl` — semua hasil (gitignored)
- `data/samples/indonli_sample.csv` — 50 baris sample (tracked di git)

**Sumber:** https://github.com/ir-nlp-csui/indonli

## 1. Setup

In [2]:
# Install jika belum ada
# !pip install pandas tqdm
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pandas', 'tqdm', '-q'])

CompletedProcess(args=['c:\\Users\\Sawawa Egatya\\miniconda3\\python.exe', '-m', 'pip', 'install', 'pandas', 'tqdm', '-q'], returncode=0)

## 2. Import & Path Configuration

In [3]:
import sys, os, json, re
from pathlib import Path
import pandas as pd
from tqdm import tqdm

REPO_ROOT   = Path('~/College/ai/FP-AI-Detecting-AI-generated-text-Bahasa-').expanduser()
sys.path.insert(0, str(REPO_ROOT))

from src.data.paths import PATHS, ensure_dirs
ensure_dirs()

# Clone target
INDONLI_DIR = REPO_ROOT / 'data' / 'raw' / 'indonli_repo'
OUTPUT_FILE = PATHS['raw_dir'] / 'indonli_human.jsonl'
SAMPLE_FILE = PATHS['samples_dir'] / 'indonli_sample.csv'

print(f'REPO_ROOT   : {REPO_ROOT}')
print(f'INDONLI_DIR : {INDONLI_DIR}')
print(f'OUTPUT_FILE : {OUTPUT_FILE}')
print(f'SAMPLE_FILE : {SAMPLE_FILE}')

ModuleNotFoundError: No module named 'src'

## 3. Clone IndoNLI Repository

In [ ]:
import subprocess

INDONLI_GITHUB = 'https://github.com/ir-nlp-csui/indonli.git'

if INDONLI_DIR.exists():
    print('✓ Repo sudah ada, pull update...')
    result = subprocess.run(['git', '-C', str(INDONLI_DIR), 'pull'],
                            capture_output=True, text=True)
else:
    print('Cloning IndoNLI repo...')
    INDONLI_DIR.parent.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(['git', 'clone', '--depth=1', INDONLI_GITHUB, str(INDONLI_DIR)],
                            capture_output=True, text=True)

print(result.stdout or result.stderr)

# Lihat isi repo
print('\nIsi folder indonli_repo:')
for f in sorted(INDONLI_DIR.rglob('*.json*')):
    print(f'  {f.relative_to(INDONLI_DIR)}')

Cloning IndoNLI repo...
Cloning into '/home/riverz/College/ai/FP-AI-Detecting-AI-generated-text-Bahasa-/data/raw/indonli_repo'...


Isi folder indonli_repo:
  data/indonli/diagnostic.jsonl
  data/indonli/test.jsonl
  data/indonli/test_expert.jsonl
  data/indonli/test_lay.jsonl
  data/indonli/train.jsonl
  data/indonli/val.jsonl
  experiments/experiments/tasks/configs/indo_nli_augment_config.json
  experiments/experiments/tasks/configs/indo_nli_config.json
  experiments/experiments/tasks/configs/indo_nli_ho_config.json
  experiments/experiments/tasks/configs/indo_xnli_config.json
  experiments/experiments/tasks/configs/indo_xnli_small_config.json
  experiments/experiments/tasks/configs/indo_xnli_small_ho_config.json
  experiments/experiments/tasks/data/indo_nli/test.jsonl
  experiments/experiments/tasks/data/indo_nli/test_expert.jsonl
  experiments/experiments/tasks/data/indo_nli/test_lay.jsonl
  experiments/experiments/tasks/data/indo_nli/train.jsonl
  experiments/experiments/tasks/dat

## 4. Load Semua Split

In [ ]:
def load_jsonl(filepath):
    """Load file JSONL atau JSON array."""
    with open(filepath, encoding='utf-8') as f:
        raw = f.read().strip()

    if raw.startswith('['):          # JSON array
        return json.loads(raw)

    records = []                     # JSONL (satu record per baris)
    for line in raw.splitlines():
        line = line.strip()
        if line:
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return records

# Filter: skip .git dan experiments
data_files = [
    f for f in INDONLI_DIR.rglob('*.jsonl')
    if '.git' not in f.parts
    and 'experiment' not in str(f).lower()
]

print(f"Found {len(data_files)} data files:")
for f in sorted(data_files):
    print(f"  {f.relative_to(INDONLI_DIR)}  ({f.stat().st_size:,} bytes)")

print("\n--- Preview baris pertama ---")
for f in sorted(data_files):
    with open(f, encoding='utf-8') as fh:
        first = fh.readline().strip()
    print(f"  {f.name}: {first[:80]}...")

# Load semua
all_records = []
for fpath in sorted(data_files):
    records = load_jsonl(fpath)      # ← konsisten, pakai load_jsonl
    print(f"  {fpath.name:<30} → {len(records):>5} rows")
    all_records.extend(records)

print(f"\nTotal semua split: {len(all_records)} rows")

Found 6 data files:
  data/indonli/diagnostic.jsonl  (284,129 bytes)
  data/indonli/test.jsonl  (2,404,282 bytes)
  data/indonli/test_expert.jsonl  (1,608,272 bytes)
  data/indonli/test_lay.jsonl  (796,010 bytes)
  data/indonli/train.jsonl  (3,788,819 bytes)
  data/indonli/val.jsonl  (784,776 bytes)

--- Preview baris pertama ---
  diagnostic.jsonl: [...
  test.jsonl: {"pair_id": 108022, "premise_id": 10802, "premise": "Kereta tersebut tiba di Gar...
  test_expert.jsonl: {"pair_id": 33321, "premise_id": 3332, "author_label": "c", "annotation_round": ...
  test_lay.jsonl: {"pair_id": 108022, "premise_id": 10802, "premise": "Kereta tersebut tiba di Gar...
  train.jsonl: {"pair_id": 101315, "premise_id": 10131, "premise": "Presiden Joko Widodo (Jokow...
  val.jsonl: {"pair_id": 119373, "premise_id": 11937, "premise": "Manuskrip tersebut berisi t...
  diagnostic.jsonl               →   650 rows
  test.jsonl                     →  5185 rows
  test_expert.jsonl              →  2984 rows
  te

## 5. Inspeksi Struktur Data

In [ ]:
print('Keys sample record:', list(all_records[0].keys()))

# Cek berapa record yang tidak punya sentence_size
missing = sum(1 for r in all_records if 'sentence_size' not in r)
print(f'Records tanpa sentence_size: {missing}/{len(all_records)}')

Keys sample record: ['pair_id', 'label', 'premise', 'hypothesis', 'inference_phenomena']
Records tanpa sentence_size: 650/23547


In [ ]:
print('Keys sample record:', list(all_records[0].keys()))
print('\nLabel distribution:', Counter(r['label'] for r in all_records))
print('Sentence sizes:', Counter(r.get('sentence_size', 'n/a') for r in all_records))

Keys sample record: ['pair_id', 'label', 'premise', 'hypothesis', 'inference_phenomena']

Label distribution: Counter({'e': 8213, 'c': 7925, 'n': 7409})
Sentence sizes: Counter({'single': 16892, 'double': 4428, 'multiple': 1577, 'n/a': 650})


## 6. Deduplikasi & Filter

Strategi:
1. Dedup berdasarkan `premise_id` — satu premise muncul berkali-kali di baris NLI yang berbeda
2. Filter panjang minimum **15 kata** — teks terlalu pendek sulit untuk AI detection
3. Hapus noise (karakter aneh, teks yang bukan Bahasa Indonesia)

In [ ]:
# Cek keys per split untuk tahu perbedaan schema
from collections import defaultdict
schema_groups = defaultdict(list)
for r in all_records:
    keys = frozenset(r.keys())
    schema_groups[keys].append(r)

for keys, records in schema_groups.items():
    print(f"{len(records):>5} records → keys: {sorted(keys)}")

  650 records → keys: ['hypothesis', 'inference_phenomena', 'label', 'pair_id', 'premise']
16929 records → keys: ['annotator_type', 'data_split', 'hypothesis', 'label', 'pair_id', 'premise', 'premise_id', 'sentence_size']
 5968 records → keys: ['annotation_round', 'annotator_type', 'author_label', 'data_split', 'hypothesis', 'label', 'pair_id', 'premise', 'premise_id', 'sentence_size', 'source', 'topics']


In [ ]:
def basic_clean(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_valid(text, min_words=15):
    if not text or len(text) < 30:
        return False
    if len(text.split()) < min_words:
        return False
    alpha_ratio = sum(c.isalpha() for c in text) / len(text)
    return alpha_ratio >= 0.6

def get_id(r):
    # test_expert tidak punya premise_id, fallback ke pair_id
    return r.get('premise_id') or r.get('pair_id')

# Dedup
seen_ids = set()
unique_records = []
for r in all_records:
    pid = get_id(r)
    if pid not in seen_ids:
        seen_ids.add(pid)
        unique_records.append(r)

print(f'Sebelum dedup : {len(all_records):>6} rows')
print(f'Setelah dedup : {len(unique_records):>6} rows')

# Filter
MIN_WORDS = 15
filtered = []
for r in tqdm(unique_records, desc='Filtering'):
    text = basic_clean(r['premise'])
    if is_valid(text, min_words=MIN_WORDS):
        filtered.append({**r, 'premise': text})

print(f'Setelah filter (>={MIN_WORDS} kata): {len(filtered):>6} rows')
print(f'Dropped: {len(unique_records) - len(filtered)} rows')

Sebelum dedup :  23547 rows
Setelah dedup :   3643 rows


Filtering: 100%|██████████| 3643/3643 [00:00<00:00, 91469.81it/s]

Setelah filter (>=15 kata):   2719 rows
Dropped: 924 rows


## 7. Eksplorasi Data (EDA)

In [ ]:
%pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 5.0 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 5.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 5.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 6.5 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib]6 [matplotlib]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

word_counts = [len(r['premise'].split()) for r in filtered]
sizes = Counter(r.get('sentence_size', 'n/a') for r in filtered)

print(f'=== STATISTIK SETELAH FILTER ===')
print(f'Total samples   : {len(filtered)}')
print(f'Word count avg  : {sum(word_counts)/len(word_counts):.1f}')
print(f'Word count min  : {min(word_counts)}')
print(f'Word count max  : {max(word_counts)}')
print(f'Sentence sizes  : {dict(sizes)}')

# Plot distribusi
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(word_counts, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribusi Panjang Premise (kata)')
axes[0].set_xlabel('Jumlah Kata')
axes[0].set_ylabel('Frekuensi')
axes[0].axvline(15, color='red', linestyle='--', label='Min 15 kata')
axes[0].legend()

axes[1].bar(sizes.keys(), sizes.values(), color=['#4C72B0','#DD8452','#55A868'])
axes[1].set_title('Distribusi Sentence Size')
axes[1].set_ylabel('Jumlah')

plt.tight_layout()
fig_path = REPO_ROOT / 'results' / 'figures' / 'indonli_eda.png'
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'\nFigure saved → {fig_path}')

=== STATISTIK SETELAH FILTER ===
Total samples   : 2719
Word count avg  : 28.0
Word count min  : 15
Word count max  : 321
Sentence sizes  : {'n/a': 538, 'single': 1531, 'multiple': 180, 'double': 470}

Figure saved → /home/riverz/College/ai/FP-AI-Detecting-AI-generated-text-Bahasa-/results/figures/indonli_eda.png


/tmp/ipykernel_39995/3804137795.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Sampling & Simpan Output

> Target: **1500 samples** (sesuai rencana dataset)

Prioritas: `sentence_size = double/multiple` dulu (teks lebih panjang), baru `single`.

In [ ]:
import random
random.seed(42)

TARGET_SAMPLES = 1500

# Prioritaskan double & multiple (lebih kaya konteks)
multi  = [r for r in filtered if r.get('sentence_size') in ('double', 'multiple')]
single = [r for r in filtered if r.get('sentence_size') not in ('double', 'multiple')]

random.shuffle(multi)
random.shuffle(single)

# Ambil semua multi, sisanya dari single
selected = multi[:TARGET_SAMPLES]
remaining = TARGET_SAMPLES - len(selected)
if remaining > 0:
    selected += single[:remaining]

random.shuffle(selected)

sizes_selected = Counter(r.get('sentence_size', 'n/a') for r in selected)
wc_selected = [len(r['premise'].split()) for r in selected]

print(f'Selected: {len(selected)} samples')
print(f'Sentence sizes: {dict(sizes_selected)}')
print(f'Word count avg: {sum(wc_selected)/len(wc_selected):.1f}')

Selected: 1500 samples
Sentence sizes: {'single': 611, 'multiple': 180, 'double': 470, 'n/a': 239}
Word count avg: 30.3


## 9. Build Output Records & Simpan

In [ ]:
output_records = []
for i, r in enumerate(selected):
    output_records.append({
        'sample_id'   : f'HUM_INDONLI_{i+1:04d}',
        'text'        : r['premise'],
        'label'       : 'human',
        'label_int'   : 0,
        'source'      : 'IndoNLI',
        'source_split': r.get('data_split', ''),
        'sentence_size': r.get('sentence_size') or r.get('inference_phenomena', 'n/a'),
        'word_count'  : len(r['premise'].split()),
        'premise_id'  : r.get('premise_id') or r.get('pair_id') or 'n/a',
    })

# Simpan ke data/raw/ (gitignored)
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for rec in output_records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print(f'✓ Saved {len(output_records)} records → {OUTPUT_FILE}')

# Simpan sample 50 rows ke data/samples/ (tracked di git)
df_sample = pd.DataFrame(output_records[:50])
df_sample.to_csv(SAMPLE_FILE, index=False)
print(f'✓ Saved sample (50 rows) → {SAMPLE_FILE}')

# Preview
print('\n--- Preview 3 records ---')
for rec in output_records[:3]:
    print(f"  [{rec['sample_id']}] ({rec['word_count']} kata) {rec['text'][:80]}...")

✓ Saved 1500 records → /home/riverz/College/ai/FP-AI-Detecting-AI-generated-text-Bahasa-/data/raw/indonli_human.jsonl
✓ Saved sample (50 rows) → /home/riverz/College/ai/FP-AI-Detecting-AI-generated-text-Bahasa-/data/samples/indonli_sample.csv

--- Preview 3 records ---
  [HUM_INDONLI_0001] (20 kata) Lincoln menjabat sebagai presiden sejak 4 April 1861. Ia dikenal sebagai sosok p...
  [HUM_INDONLI_0002] (63 kata) Aceh Darussalam pada zaman kekuasaan zaman Sultan Iskandar Muda Meukuta Perkasa ...
  [HUM_INDONLI_0003] (26 kata) Berbeda dari tahun lalu, musim ini di ajang Premier Badminton League (PBL) 2020,...


## 10. Verifikasi Output

In [ ]:
# Load balik dan cek
verify = []
with open(OUTPUT_FILE, encoding='utf-8') as f:
    for line in f:
        verify.append(json.loads(line))

df_verify = pd.DataFrame(verify)
print('=== VERIFIKASI ===')
print(df_verify[['label','word_count','sentence_size']].describe(include='all').to_string())
print(f'\nLabel unique  : {df_verify["label"].unique()}')
print(f'label_int     : {df_verify["label_int"].unique()}')
print(f'Total records : {len(df_verify)}')
print(f'\n✅ IndoNLI human data siap digunakan!')
print(f'   Next step: gabungkan dengan ai_generated_gemini.jsonl + ai_generated_openai.jsonl')
print(f'   di script src/data/prepare_dataset.py')

=== VERIFIKASI ===
        label   word_count sentence_size
count    1500  1500.000000          1500
unique      1          NaN           130
top     human          NaN        single
freq     1500          NaN           611
mean      NaN    30.252000           NaN
std       NaN    17.942067           NaN
min       NaN    15.000000           NaN
25%       NaN    18.000000           NaN
50%       NaN    24.000000           NaN
75%       NaN    35.000000           NaN
max       NaN   130.000000           NaN

Label unique  : <ArrowStringArray>
['human']
Length: 1, dtype: str
label_int     : [0]
Total records : 1500

✅ IndoNLI human data siap digunakan!
   Next step: gabungkan dengan ai_generated_gemini.jsonl + ai_generated_openai.jsonl
   di script src/data/prepare_dataset.py
